In [1]:
import torch
import torchaudio
import whisper
import onnxruntime
import numpy as np

def extract_speech_token(audio_path, onnx_model_path):
    """
    提取单个音频文件的 speech token
    :param audio_path: 音频文件路径
    :param onnx_model_path: ONNX 模型路径
    :return: speech token (numpy array)
    """
    # 加载音频文件
    audio, sample_rate = torchaudio.load(audio_path)

    # 如果采样率不是16000，则进行重采样
    if sample_rate != 16000:
        audio = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)(audio)

    # 如果音频长度超过30秒，则返回空列表
    if audio.shape[1] / 16000 > 30:
        print(f'音频 {audio_path} 超过30秒，不支持提取语音特征')
        return []

    # 计算音频的梅尔频谱图
    feat = whisper.log_mel_spectrogram(audio, n_mels=128)

    # 配置ONNX Runtime会话
    option = onnxruntime.SessionOptions()
    option.graph_optimization_level = onnxruntime.GraphOptimizationLevel.ORT_ENABLE_ALL
    ort_session = onnxruntime.InferenceSession(onnx_model_path, sess_options=option)

    # 获取输入名称
    input_name = ort_session.get_inputs()[0].name
    input_length_name = ort_session.get_inputs()[1].name

    # 使用ONNX模型进行推理
    outputs = ort_session.run(None, {
        input_name: feat.detach().cpu().numpy(),
        input_length_name: np.array([feat.shape[2]], dtype=np.int32)
    })

    # 返回 speech token
    return outputs[0].flatten()


In [2]:
import torch
import sys

sys.path.append('/root/LLMs/CosyVoice/third_party/Matcha-TTS')
from matcha.utils.audio import mel_spectrogram

# sampling_rate = 22050

class Feat_extractor:

    def __init__(self, n_fft: int = 1920, num_mels: int = 80, sampling_rate: int = 24000, 
                 hop_size: int = 480, win_size: int = 1920, fmin: int = 0, fmax: int = 8000, 
                 center: bool = False, device: str = 'cuda'):
        self.device = torch.device(device)
        self.feat_extractor = lambda x: mel_spectrogram(
            x, n_fft=n_fft, num_mels=num_mels, sampling_rate=sampling_rate, 
            hop_size=hop_size, win_size=win_size, fmin=fmin, fmax=fmax, center=center
        )

    def __call__(self, speech):
        speech_feat = self.feat_extractor(speech).squeeze(dim=0).transpose(0, 1).to(self.device)
        speech_feat = speech_feat.unsqueeze(dim=0)
        speech_feat_len = torch.tensor([speech_feat.shape[1]], dtype=torch.int32).to(self.device)
        return speech_feat, speech_feat_len


/root/miniconda3/envs/Cosyvoice/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import torch
import torchaudio
import torchaudio.compliance.kaldi as kaldi
import onnxruntime
import numpy as np

def extract_embedding(audio_path, onnx_model_path):
    """
    提取单个音频文件的 embedding
    :param audio_path: 音频文件路径
    :param onnx_model_path: ONNX 模型路径
    :return: embedding (numpy array)
    """
    # 加载音频文件
    audio, sample_rate = torchaudio.load(audio_path)

    # 如果采样率不是16000，则进行重采样
    if sample_rate != 16000:
        audio = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)(audio)

    # 计算音频的FBank特征
    feat = kaldi.fbank(
        audio,
        num_mel_bins=80,  # 梅尔频带数
        dither=0,         # 禁用抖动
        sample_frequency=16000
    )

    # 特征归一化
    feat = feat - feat.mean(dim=0, keepdim=True)

    # 配置ONNX Runtime会话
    option = onnxruntime.SessionOptions()
    option.graph_optimization_level = onnxruntime.GraphOptimizationLevel.ORT_ENABLE_ALL
    ort_session = onnxruntime.InferenceSession(onnx_model_path, sess_options=option)

    # 使用ONNX模型进行推理
    input_name = ort_session.get_inputs()[0].name
    embedding = ort_session.run(None, {input_name: feat.unsqueeze(dim=0).cpu().numpy()})[0]

    # 返回 embedding
    return embedding.flatten()

In [4]:
import sys

sys.path.append('/root/LLMs/CosyVoice/third_party/Matcha-TTS')
sys.path.append('/root/LLMs/CosyVoice')

from cosyvoice.utils.file_utils import load_wav
import torchaudio

voice_tokenizer_path = '/root/LLMs/CosyVoice/pretrained_models/CosyVoice-300M/speech_tokenizer_v1.onnx'
embedding_path = '/root/LLMs/CosyVoice/pretrained_models/CosyVoice-300M/campplus.onnx'



input_speech_16k_path = '/root/LLMs/Sushi_Series/SpeechLM/GLM Voice/如果你也听说_mono_17s.wav'
input_speech_16k = load_wav(input_speech_16k_path, 16000)
# input_speech_16k_path = '/root/LLMs/CosyVoice/data/zero_shot_prompt.wav'
# 提取语音 token
input_token = extract_speech_token(input_speech_16k_path, voice_tokenizer_path)
input_token = torch.tensor(input_token).unsqueeze(0)




In [22]:
resample_rate = 16000

# 提取 mel 特征
front_end = Feat_extractor()

input_speech_resample = torchaudio.transforms.Resample(orig_freq=32000, new_freq=resample_rate)(input_speech_16k)
input_feat, input_feat_len = front_end(input_speech_resample)

# 提取 speaker embedding
embedding = extract_embedding(input_speech_16k_path, embedding_path)
embedding = torch.tensor(embedding).unsqueeze(0).to('cuda')

min value is  tensor(-1.1116)
max value is  tensor(1.2323)


In [23]:
input_feat.shape

torch.Size([1, 288, 80])

In [24]:
embedding.shape

torch.Size([1, 192])

In [5]:
import torch
from cosyvoice.cli.cosyvoice import CosyVoice
from cosyvoice.utils.file_utils import load_wav
import torchaudio

cosyvoice = CosyVoice('/root/LLMs/CosyVoice/pretrained_models/CosyVoice-300M') 

2025-01-23 22:50:24,518 - modelscope - INFO - PyTorch version 2.3.1+cu121 Found.
2025-01-23 22:50:24,520 - modelscope - INFO - Loading ast index from /root/.cache/modelscope/ast_indexer
2025-01-23 22:50:24,628 - modelscope - INFO - Loading done! Current index file version is 1.15.0, with md5 bb0aa2b4cdc4ce378b905081ade43db6 and a total number of 980 components indexed


failed to import ttsfrd, use WeTextProcessing instead


/root/miniconda3/envs/Cosyvoice/lib/python3.10/site-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)
2025-01-23 22:50:28,634 INFO input frame rate=50
2025-01-23 22:50:29.961479704 [W:onnxruntime:, transformer_memcpy.cc:74 ApplyImpl] 12 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
2025-01-23 22:50:29.962536690 [W:onnxruntime:, session_state.cc:1166 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape

In [30]:
import torch
import IPython.display as ipd
import uuid

this_uuid = str(uuid.uuid1())
cosyvoice.model.flow_cache_dict[this_uuid] = torch.zeros(1, 80, 0, 2)
cosyvoice.model.hift_cache_dict[this_uuid] = None
cosyvoice.model.mel_overlap_dict[this_uuid] = torch.zeros(1, 80, 0)

prompt_token=torch.zeros(1, 0, dtype=torch.int32)
prompt_feat=torch.zeros(1, 0, 80)
embedding=torch.zeros(1, 192)

this_tts_speech = cosyvoice.model.token2wav(token=input_token,
                                            prompt_token=prompt_token,#speech_token,
                                            prompt_feat=prompt_feat,#speech_feat,
                                            embedding=embedding,
                                            uuid=this_uuid,
                                            finalize=True,
                                            speed=1)

sr = 22400
tts_speech = this_tts_speech.cpu()
# 在 Jupyter Notebook 中播放音频
ipd.Audio(tts_speech, rate=sr)